# WikiRisk — Full ML Training Pipeline

**Dataset:** Wikimedia `mediawiki_history` revision dumps (2023–2024)  
**Label:** `revision_is_identity_reverted` — community ground truth (edit was reverted by other editors = damaging)  
**Model:** SparkML Logistic Regression with TF-IDF text features + numeric features  
**MLOps:** MLflow experiment tracking + model registration  

---

## 1. Setup

In [ ]:
import os, sys, time, json, re, sqlite3, urllib.request, warnings
from pathlib import Path
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
warnings.filterwarnings('ignore')

ROOT       = Path('..').resolve()
DATA_DIR   = ROOT / 'data' / 'raw' / 'mediawiki_history'
MODEL_DIR  = ROOT / 'data' / 'model'
MANIFEST   = ROOT / 'manifests' / 'model_manifest.json'
MLFLOW_DIR = ROOT / 'mlruns'
DB_PATH    = ROOT / 'data' / 'wikirisk.db'

for d in [DATA_DIR, MODEL_DIR, ROOT/'manifests', ROOT/'mlruns']:
    d.mkdir(parents=True, exist_ok=True)

print('ROOT   :', ROOT)
print('DATA   :', DATA_DIR)
print('Python :', sys.version.split()[0])

import pyspark, mlflow
print('PySpark:', pyspark.__version__)
print('MLflow :', mlflow.__version__)

## 2. Download — Wikimedia mediawiki_history Dumps

Two monthly snapshots covering **2023** and **2024** English Wikipedia edit history.  
Each file is ~600–830 MB compressed / ~5–7 GB uncompressed (bz2 ratio ≈ 9×).  
**Total: ~12 GB uncompressed — satisfies the 10+ GB data requirement.**

In [ ]:
BASE_URL = 'https://dumps.wikimedia.org/other/mediawiki_history/2026-03/enwiki'
FILES    = [
    '2026-03.enwiki.2022-01.tsv.bz2',
    '2026-03.enwiki.2023-01.tsv.bz2',
    '2026-03.enwiki.2024-01.tsv.bz2',
]

def fmt_bytes(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} TB'

downloaded = []
for fname in FILES:
    dest = DATA_DIR / fname
    url  = f'{BASE_URL}/{fname}'
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f'✅ Already on disk: {fname}  ({fmt_bytes(dest.stat().st_size)})')
        downloaded.append(dest)
        continue
    print(f'⬇️  Downloading {fname} ...')
    req = urllib.request.Request(url, headers={'User-Agent': 'WikiRisk/1.0 (academic)'})
    t0  = time.monotonic()
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, 'wb') as fh:
        total = int(r.headers.get('Content-Length', 0))
        done  = 0
        while chunk := r.read(1 << 20):
            fh.write(chunk); done += len(chunk)
            print(f'\r   {fmt_bytes(done)}/{fmt_bytes(total)} '
                  f'({100*done/total:.1f}%)  '
                  f'{fmt_bytes(done/(time.monotonic()-t0))}/s   ', end='')
    elapsed = time.monotonic() - t0
    print(f'\n   Done in {elapsed:.0f}s  →  {fmt_bytes(dest.stat().st_size)}')
    downloaded.append(dest)

total_compressed = sum(f.stat().st_size for f in downloaded)
print(f'\n📦 Total compressed : {fmt_bytes(total_compressed)}')
print(f'✅ Files ready      : {[f.name for f in downloaded]}')

# Measure exact uncompressed size — cached so we never decompress twice
import bz2
SIZE_CACHE = DATA_DIR / 'sizes.json'

if SIZE_CACHE.exists():
    cached = json.loads(SIZE_CACHE.read_text())
    print('\n✅ Uncompressed sizes (from cache):')
    total_uncompressed = 0
    for f in downloaded:
        size = cached.get(f.name, 0)
        total_uncompressed += size
        print(f'   {f.name}: {fmt_bytes(size)}')
else:
    print('\n⏳ Measuring exact uncompressed size (streaming bz2 decode, runs once)...')
    sizes_map = {}
    total_uncompressed = 0
    for f in downloaded:
        size = 0
        with bz2.open(f, 'rb') as fh:
            while chunk := fh.read(1 << 20):
                size += len(chunk)
        sizes_map[f.name] = size
        total_uncompressed += size
        print(f'   {f.name}: {fmt_bytes(size)} uncompressed')
    SIZE_CACHE.write_text(json.dumps(sizes_map, indent=2))
    print(f'   (sizes cached to {SIZE_CACHE.name} — won\'t re-decompress next run)')

print(f'\n✅ EXACT total uncompressed: {total_uncompressed:,} bytes  =  {total_uncompressed/1024**3:.2f} GB')

## 3. Start SparkSession

In [ ]:
STATS_CACHE   = DATA_DIR / 'label_stats.json'
_model_exists = (MODEL_DIR / 'wikirisk_lr_model').exists() and MANIFEST.exists()
_stats_exists = STATS_CACHE.exists()
_skip_spark   = _model_exists and _stats_exists

if _skip_spark:
    # Load everything from cache — no Spark needed
    label_stats = json.loads(STATS_CACHE.read_text())
    saved       = json.loads(MANIFEST.read_text())
    total  = label_stats['total']
    pos    = label_stats['positive']
    neg    = label_stats['negative']
    n_train = label_stats.get('n_train', 0)
    n_test  = label_stats.get('n_test',  0)
    metrics = saved.get('metrics', {})
    RUN_ID  = saved.get('run_id', 'cached')
    model_path = str(MODEL_DIR / 'wikirisk_lr_model')
    elapsed = metrics.get('training_time_seconds', 0)
    print('✅ Model + stats loaded from cache — Spark data pipeline SKIPPED.')
    print(f'   Dataset  : {total:,} revisions  ({100*pos/total:.1f}% reverted)')
    print(f'   Trained at : {saved.get("trained_at","?")}')
    print(f'   AUC-ROC  : {metrics.get("auc_roc","?")}   F1: {metrics.get("f1","?")}')
    spark = None  # Will start lazily only if needed below
else:
    print('🚀 First run — starting SparkSession for full pipeline...')
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .master('local[*]')
        .appName('WikiRisk-Training')
        .config('spark.driver.memory',          '6g')
        .config('spark.executor.memory',        '6g')
        .config('spark.sql.shuffle.partitions', '100')
        .config('spark.ui.enabled',             'false')
        .config('spark.sql.adaptive.enabled',   'true')
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel('WARN')
    print(f'Spark {spark.version} — {os.cpu_count()} cores')

## 4. Load Raw Data

The TSV files have 56 columns. We read them all as strings and cast the ones we need.

In [ ]:
if _skip_spark:
    print('✅ Skipping raw data load — model already trained, stats cached.')
    raw = labeled = train_df = test_df = None
    total_raw = label_stats['total']
else:
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, StructField, StringType

    MW_COLUMNS = [
        'wiki_db','event_entity','event_type','event_timestamp',
        'event_comment','event_user_id','event_user_text_historical',
        'event_user_text','event_user_blocks_historical','event_user_blocks',
        'event_user_groups_historical','event_user_groups',
        'event_user_is_bot_by_historical','event_user_is_bot_by',
        'event_user_is_created_by_self','event_user_is_created_by_system',
        'event_user_is_created_by_peer','event_user_is_anonymous',
        'event_user_registration_timestamp','event_user_creation_timestamp',
        'event_user_first_edit_timestamp','event_user_revision_count',
        'event_user_seconds_since_previous_revision',
        'page_id','page_title_historical','page_title',
        'page_namespace_historical','page_namespace',
        'page_namespace_is_content_historical','page_namespace_is_content',
        'page_is_redirect_historical','page_is_redirect',
        'page_is_deleted','page_creation_timestamp',
        'page_first_edit_timestamp','page_revision_count',
        'page_seconds_since_previous_revision',
        'revision_id','revision_parent_id','revision_minor_edit',
        'revision_deleted_parts','revision_deleted_parts_are_suppressed',
        'revision_text_bytes','revision_text_bytes_diff',
        'revision_text_sha1','revision_content_model',
        'revision_content_format','revision_is_deleted_by_page_deletion',
        'revision_deleted_timestamp','revision_is_identity_reverted',
        'revision_first_identity_reverting_revision_id',
        'revision_seconds_to_identity_revert','revision_is_identity_revert',
        'revision_is_from_before_page_creation','revision_tags',
    ]

    schema = StructType([StructField(c, StringType(), True) for c in MW_COLUMNS])
    paths  = [str(f) for f in downloaded]

    t0  = time.monotonic()
    raw = (
        spark.read
        .option('sep', '\t').option('header', 'false')
        .option('quote', '').option('comment', '#')
        .schema(schema).csv(paths)
    )
    raw.cache()
    total_raw = raw.count()
    print(f'⏱  Load + count: {time.monotonic()-t0:.1f}s')
    print(f'📄 Total raw rows : {total_raw:,}')
    print(f'📋 Columns        : {len(raw.columns)}')

## 5. Inspect Raw Data

In [ ]:
if _skip_spark:
    print('✅ Skipping raw data inspection (cached)')
else:
    print('Event entity breakdown:')
    raw.groupBy('event_entity').count().orderBy('count', ascending=False).show(10)
    print('\nSample row (key fields):')
    raw.select('event_entity','event_type','page_namespace',
               'event_comment','revision_is_identity_reverted',
               'revision_text_bytes_diff').show(3, truncate=60)

## 6. Filter + Engineer Features

Keep only **article-space revision creates** (namespace=0, entity=revision, type=create).  
Label = `revision_is_identity_reverted` — edits that were subsequently reverted by the Wikipedia community.

In [ ]:
if _skip_spark:
    print('✅ Skipping feature engineering — using cached label stats:')
    total = label_stats['total']; pos = label_stats['positive']; neg = label_stats['negative']
    rate  = label_stats['positive_rate']
    print(f'   Total labeled : {total:,}')
    print(f'   Reverted (1)  : {pos:,}  ({100*rate:.1f}%)')
    print(f'   Benign   (0)  : {neg:,}  ({100*(1-rate):.1f}%)')
else:
    t0 = time.monotonic()
    labeled = (
        raw
        .filter(F.col('event_entity') == 'revision')
        .filter(F.col('event_type')   == 'create')
        .filter(F.col('page_namespace') == '0')
        .filter(F.col('revision_is_identity_reverted').isNotNull())
        .withColumn('label',
            (F.col('revision_is_identity_reverted') == 'true').cast('int'))
        .withColumn('length_delta',
            F.coalesce(F.col('revision_text_bytes_diff').cast('double'), F.lit(0.0)))
        .withColumn('is_anon_int',
            (F.col('event_user_is_anonymous') == 'true').cast('int'))
        .withColumn('event_ts',
            F.to_timestamp('event_timestamp', 'yyyy-MM-dd HH:mm:ss.S'))
        .withColumn('hour_of_day', F.hour('event_ts'))
        .withColumn('day_of_week', F.dayofweek('event_ts'))
        .withColumn('comment_clean',
            F.lower(F.regexp_replace(
                F.coalesce(F.col('event_comment'), F.lit('')), r'[^a-zA-Z0-9\s]', ' ')))
        .withColumn('title_clean',
            F.lower(F.regexp_replace(
                F.coalesce(F.col('page_title_historical'), F.lit('')), r'[^a-zA-Z0-9\s]', ' ')))
        .select('revision_id','page_title_historical','label',
                'length_delta','is_anon_int','hour_of_day','day_of_week',
                'comment_clean','title_clean')
        .na.fill({'length_delta':0.0,'is_anon_int':0,'hour_of_day':0,
                  'day_of_week':1,'comment_clean':'','title_clean':''})
    )
    labeled.cache()
    total = labeled.count()
    pos   = labeled.filter(F.col('label') == 1).count()
    neg   = total - pos
    rate  = pos / total
    print(f'⏱  Filter + feature engineering: {time.monotonic()-t0:.1f}s')
    print(f'📊 Total labeled revisions : {total:,}')
    print(f'   Label=1 (reverted/risky): {pos:,}  ({100*rate:.1f}%)')
    print(f'   Label=0 (benign)         : {neg:,}  ({100*(1-rate):.1f}%)')
    label_stats = {'total':total,'positive':pos,'negative':neg,'positive_rate':round(rate,4)}
    # Cache stats so subsequent runs skip Spark entirely
    STATS_CACHE.write_text(json.dumps(label_stats, indent=2))
    print(f'   Stats cached to {STATS_CACHE.name}')

## 7. Data Exploration

In [ ]:
if _skip_spark:
    print('✅ Skipping sample display (cached)')
else:
    print('Sample labeled records:')
    labeled.show(5, truncate=70)

In [ ]:
_explore_png = ROOT / 'docs' / 'dataset_exploration.png'
if _skip_spark and _explore_png.exists():
    print('✅ Loading saved exploration chart:')
    from IPython.display import Image, display
    display(Image(str(_explore_png)))
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Dataset Exploration — Wikimedia History (2022–2024)', fontsize=14, fontweight='bold')
    axes[0].bar(['Benign (0)', 'Reverted (1)'], [neg, pos],
                color=['#22c55e', '#dc2626'], edgecolor='white', linewidth=0.5)
    axes[0].set_title('Label Distribution'); axes[0].set_ylabel('Revisions')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    for bar, val in zip(axes[0].patches, [neg, pos]):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01, f'{val:,}', ha='center', va='bottom', fontsize=9)
    anon_dist = labeled.groupBy('is_anon_int').count().toPandas()
    anon_dist['label_str'] = anon_dist['is_anon_int'].map({0: 'Registered', 1: 'Anonymous'})
    axes[1].bar(anon_dist['label_str'], anon_dist['count'], color=['#3b82f6', '#f59e0b'], edgecolor='white')
    axes[1].set_title('Editor Type'); axes[1].set_ylabel('Revisions')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    hour_dist = labeled.groupBy('hour_of_day').count().orderBy('hour_of_day').toPandas()
    axes[2].bar(hour_dist['hour_of_day'], hour_dist['count'], color='#6366f1', edgecolor='white', linewidth=0.3)
    axes[2].set_title('Edit Activity by Hour (UTC)'); axes[2].set_xlabel('Hour'); axes[2].set_ylabel('Revisions')
    axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
    plt.tight_layout()
    (ROOT / 'docs').mkdir(exist_ok=True)
    plt.savefig(_explore_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved to docs/dataset_exploration.png')

## 8. Train / Test Split

In [ ]:
if _skip_spark:
    n_train = label_stats.get('n_train', 0)
    n_test  = label_stats.get('n_test',  0)
    print(f'✅ Skipping split — cached: train={n_train:,}  test={n_test:,}')
else:
    train_df, test_df = labeled.randomSplit([0.8, 0.2], seed=42)
    train_df.cache(); test_df.cache()
    n_train = train_df.count(); n_test = test_df.count()
    print(f'Train : {n_train:,}  ({100*n_train/(n_train+n_test):.0f}%)')
    print(f'Test  : {n_test:,}   ({100*n_test/(n_train+n_test):.0f}%)')
    train_pos = train_df.filter(F.col('label')==1).count()
    test_pos  = test_df.filter(F.col('label')==1).count()
    print(f'Train positive rate: {100*train_pos/n_train:.2f}%')
    print(f'Test  positive rate: {100*test_pos/n_test:.2f}%')
    # Persist split sizes into stats cache
    label_stats.update({'n_train': n_train, 'n_test': n_test})
    STATS_CACHE.write_text(json.dumps(label_stats, indent=2))

## 9. Build SparkML Pipeline

**Features:**
- TF-IDF on edit comment (256 hash buckets)
- TF-IDF on page title (128 hash buckets)
- `length_delta` — bytes added/removed
- `is_anon_int` — anonymous editor flag
- `hour_of_day`, `day_of_week` — temporal signals

**Classifier:** Logistic Regression (interpretable, fast, proven for sparse TF-IDF features)

In [ ]:
if _skip_spark:
    print('✅ Skipping pipeline build — model already trained.')
    cv = evaluator = None
else:
    from pyspark.ml import Pipeline
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
    from pyspark.ml.feature import HashingTF, IDF, RegexTokenizer, StandardScaler, VectorAssembler
    from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

    comment_tok = RegexTokenizer(inputCol='comment_clean', outputCol='comment_tokens',
                                  pattern=r'\s+', toLowercase=True, minTokenLength=2)
    title_tok   = RegexTokenizer(inputCol='title_clean',   outputCol='title_tokens',
                                  pattern=r'\s+', toLowercase=True, minTokenLength=2)
    comment_tf  = HashingTF(inputCol='comment_tokens', outputCol='comment_tf',  numFeatures=256)
    title_tf    = HashingTF(inputCol='title_tokens',   outputCol='title_tf',    numFeatures=128)
    comment_idf = IDF(inputCol='comment_tf', outputCol='comment_idf', minDocFreq=5)
    title_idf   = IDF(inputCol='title_tf',   outputCol='title_idf',   minDocFreq=5)
    assembler   = VectorAssembler(
        inputCols=['comment_idf','title_idf','length_delta','is_anon_int','hour_of_day','day_of_week'],
        outputCol='raw_features', handleInvalid='keep')
    scaler = StandardScaler(inputCol='raw_features', outputCol='features', withMean=False, withStd=True)
    lr     = LogisticRegression(featuresCol='features', labelCol='label',
                                probabilityCol='probability', rawPredictionCol='raw_prediction')
    pipeline = Pipeline(stages=[comment_tok, title_tok, comment_tf, title_tf,
                                 comment_idf, title_idf, assembler, scaler, lr])
    param_grid = (ParamGridBuilder()
                  .addGrid(lr.regParam, [0.001, 0.01, 0.1])
                  .addGrid(lr.maxIter,  [50, 100]).build())
    evaluator = BinaryClassificationEvaluator(
        labelCol='label', rawPredictionCol='raw_prediction', metricName='areaUnderROC')
    cv = CrossValidator(estimator=pipeline, estimatorParamMaps=param_grid,
                        evaluator=evaluator, numFolds=3, seed=42)
    print(f'Pipeline stages : {[s.__class__.__name__ for s in pipeline.getStages()]}')
    print(f'Grid size       : {len(param_grid)} combinations  ×  3 folds  =  {3*len(param_grid)} fits')

## 10. Train with MLflow Tracking

In [ ]:
import mlflow, mlflow.spark
from pyspark.ml import PipelineModel as _PM

mlflow.set_tracking_uri(str(MLFLOW_DIR))
mlflow.set_experiment('wikirisk-edit-risk')

model_path = str(MODEL_DIR / 'wikirisk_lr_model')
_model_exists = (MODEL_DIR / 'wikirisk_lr_model').exists()

if _model_exists and MANIFEST.exists():
    print('✅ Trained model already exists — loading from disk (skipping re-train).')
    best_model = _PM.load(model_path)
    saved      = json.loads(MANIFEST.read_text())
    RUN_ID     = saved.get('run_id', 'loaded-from-disk')
    elapsed    = saved.get('metrics', {}).get('training_time_seconds', 0)
    best_lr    = best_model.stages[-1]
    print(f'   Model path : {model_path}')
    print(f'   MLflow run : {RUN_ID}')
    print(f'   Trained at : {saved.get("trained_at", "unknown")}')
else:
    print('🚀 No trained model found — starting cross-validation training...')
    print('   (30–60 minutes on a laptop)')
    t0 = time.monotonic()

    with mlflow.start_run(run_name='wikirisk-lr-mediawiki-history') as run:
        RUN_ID = run.info.run_id
        mlflow.log_params({
            'model_type':    'LogisticRegression',
            'data_source':   'mediawiki_history',
            'label_field':   'revision_is_identity_reverted',
            'train_size':    n_train,
            'test_size':     n_test,
            'positive_rate': label_stats['positive_rate'],
            'cv_folds':      3,
            'spark_version': spark.version,
        })
        mlflow.log_params(label_stats)

        cv_model   = cv.fit(train_df)
        elapsed    = time.monotonic() - t0
        best_model = cv_model.bestModel
        best_lr    = best_model.stages[-1]

        mlflow.log_params({
            'best_reg_param': best_lr.getRegParam(),
            'best_max_iter':  best_lr.getMaxIter(),
        })
    print(f'✅ Training done in {elapsed:.0f}s  ({elapsed/60:.1f} min)')
    print(f'   Best regParam = {best_lr.getRegParam()}  |  maxIter = {best_lr.getMaxIter()}')

## 11. Evaluate on Test Set

In [ ]:
if _skip_spark:
    print('✅ Metrics loaded from manifest (skipping re-evaluation):')
    print('=== Model Evaluation on Test Set ===')
    for k, v in metrics.items():
        print(f'  {k:<30} {v}')
else:
    preds = best_model.transform(test_df)
    from pyspark.ml.evaluation import MulticlassClassificationEvaluator as _MCE
    mc = _MCE(labelCol='label', predictionCol='prediction')
    pr = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='raw_prediction',
                                        metricName='areaUnderPR')
    metrics = {
        'auc_roc'  : round(evaluator.evaluate(preds), 4),
        'auc_pr'   : round(pr.evaluate(preds),        4),
        'f1'       : round(mc.setMetricName('f1').evaluate(preds), 4),
        'precision': round(mc.setMetricName('weightedPrecision').evaluate(preds), 4),
        'recall'   : round(mc.setMetricName('weightedRecall').evaluate(preds), 4),
        'accuracy' : round(mc.setMetricName('accuracy').evaluate(preds), 4),
        'training_time_seconds': round(elapsed, 1),
    }
    with mlflow.start_run(run_id=RUN_ID):
        mlflow.log_metrics(metrics)
    print('=== Model Evaluation on Test Set ===')
    for k, v in metrics.items():
        print(f'  {k:<30} {v}')

In [ ]:
_eval_png = ROOT / 'docs' / 'model_evaluation.png'
if _skip_spark and _eval_png.exists():
    print('✅ Loading saved evaluation chart:')
    from IPython.display import Image, display
    display(Image(str(_eval_png)))
else:
    # Build metrics bar chart from manifest/metrics dict (works even if skipping Spark)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')
    if not _skip_spark:
        cm_df = (preds.select('label','prediction')
                 .groupBy('label','prediction').count()
                 .orderBy('label','prediction').toPandas())
        cm = np.zeros((2,2), dtype=int)
        for _, row in cm_df.iterrows():
            cm[int(row['label'])][int(row['prediction'])] = int(row['count'])
        sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
                    xticklabels=['Pred Benign','Pred Risky'],
                    yticklabels=['Actual Benign','Actual Risky'], ax=axes[0])
        axes[0].set_title('Confusion Matrix')
    else:
        axes[0].text(0.5, 0.5, 'Confusion matrix\n(available on first run)',
                     ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_title('Confusion Matrix')
    mnames = ['AUC-ROC','AUC-PR','F1','Precision','Recall','Accuracy']
    mvals  = [metrics.get('auc_roc',0), metrics.get('auc_pr',0), metrics.get('f1',0),
              metrics.get('precision',0), metrics.get('recall',0), metrics.get('accuracy',0)]
    bars = axes[1].barh(mnames, mvals, color='#6366f1', edgecolor='white')
    axes[1].set_xlim(0, 1.15); axes[1].set_title('Performance Metrics')
    for bar, val in zip(bars, mvals):
        axes[1].text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    (ROOT/'docs').mkdir(exist_ok=True)
    plt.savefig(_eval_png, dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved to docs/model_evaluation.png')

## 12. Save Model Locally + Register in MLflow

In [ ]:
if _model_exists and MANIFEST.exists():
    print(f'✅ Model already saved at: {model_path}  (skipping overwrite)')
else:
    best_model.write().overwrite().save(model_path)
    print(f'✅ Model saved locally: {model_path}')
    with mlflow.start_run(run_id=RUN_ID):
        mlflow.spark.log_model(
            spark_model=best_model,
            artifact_path='spark-model',
            registered_model_name='wikirisk-risk-classifier',
        )
    print('✅ Model registered in MLflow as: wikirisk-risk-classifier')

manifest = {
    'run_id':            RUN_ID,
    'model_name':        'wikirisk-risk-classifier',
    'model_local_path':  model_path,
    'trained_at':        datetime.now(tz=timezone.utc).isoformat(),
    'data_source':       'mediawiki_history',
    'label_field':       'revision_is_identity_reverted',
    'files_used':        [f.name for f in downloaded],
    'metrics':           metrics,
    'training_data_stats': label_stats,
}
MANIFEST.write_text(json.dumps(manifest, indent=2))
print(f'✅ Manifest written: {MANIFEST}')
print(json.dumps(manifest, indent=2))

## 13. Re-score Existing Dashboard Edits

Apply the newly trained model to the 170 edits already in the SQLite DB so the dashboard immediately shows real ML scores.

In [ ]:
from pyspark.ml import PipelineModel
from pyspark.sql.types import DoubleType, IntegerType, StringType as _ST, StructField, StructType
from pyspark.sql.functions import udf as _udf

if not DB_PATH.exists():
    print('ℹ️  No dashboard database found — skipping re-score (run the API first to populate it).')
else:
    conn = sqlite3.connect(str(DB_PATH))
    already_scored = conn.execute(
        'SELECT COUNT(*) FROM predictions WHERE scored=1 AND risk_score IS NOT NULL'
    ).fetchone()[0]
    total_edits = conn.execute('SELECT COUNT(*) FROM predictions').fetchone()[0]

    if already_scored == total_edits and total_edits > 0:
        print(f'✅ All {total_edits} dashboard edits already ML-scored — skipping re-score.')
        dist = conn.execute(
            'SELECT risk_label, COUNT(*) n, ROUND(AVG(risk_score),3) avg '
            'FROM predictions GROUP BY risk_label ORDER BY avg DESC'
        ).fetchall()
        conn.close()
        print('Current distribution:')
        for r in dist:
            print(f'  {r[0]:6s}: {r[1]:3d} edits   avg score = {r[2]}')
    else:
        rows = conn.execute(
            'SELECT id, comment, page_title, length_delta, is_anon, timestamp FROM predictions'
        ).fetchall()
        conn.close()
        print(f'🔄 Re-scoring {len(rows)} existing dashboard edits...')

        schema_rescore = StructType([
            StructField('id',            _ST(),  False),
            StructField('comment_clean', _ST(),  True),
            StructField('title_clean',   _ST(),  True),
            StructField('length_delta',  DoubleType(),  True),
            StructField('is_anon_int',   IntegerType(), True),
            StructField('hour_of_day',   IntegerType(), True),
            StructField('day_of_week',   IntegerType(), True),
        ])
        records = []
        for r in rows:
            rid, comment, title, delta, is_anon, ts_str = r
            try:
                dt = datetime.fromisoformat(str(ts_str).replace('Z', '+00:00'))
                hour = dt.hour; dow = dt.isoweekday()
            except Exception:
                hour, dow = 0, 1
            records.append((
                rid,
                re.sub(r'[^a-zA-Z0-9\s]', ' ', (comment or '').lower()),
                re.sub(r'[^a-zA-Z0-9\s]', ' ', (title   or '').lower()),
                float(delta or 0), int(is_anon or 0), hour, dow,
            ))

        # Ensure Spark is running (lazy start for rescore-only runs)
        if spark is None:
            from pyspark.sql import SparkSession as _SS
            spark = (_SS.builder.master('local[2]').appName('WikiRisk-Rescore')
                     .config('spark.driver.memory','2g').config('spark.ui.enabled','false')
                     .getOrCreate())
            spark.sparkContext.setLogLevel('WARN')

        loaded = PipelineModel.load(model_path)
        df_rs  = spark.createDataFrame(records, schema=schema_rescore)
        get_prob = _udf(lambda v: float(v[1]) if v is not None else 0.5, DoubleType())
        from pyspark.sql import functions as F
        scored_df = loaded.transform(df_rs).withColumn('risk_score', get_prob(F.col('probability')))
        results   = scored_df.select('id', 'risk_score').toPandas()

        updates = []
        for _, row in results.iterrows():
            sc  = float(row['risk_score'])
            lbl = 'HIGH' if sc >= 0.68 else ('MEDIUM' if sc >= 0.30 else 'LOW')
            updates.append((sc, lbl, row['id']))

        conn = sqlite3.connect(str(DB_PATH))
        conn.executemany(
            'UPDATE predictions SET risk_score=?, risk_label=?, scored=1 WHERE id=?', updates)
        conn.commit()
        dist = conn.execute(
            'SELECT risk_label, COUNT(*) n, ROUND(AVG(risk_score),3) avg '
            'FROM predictions GROUP BY risk_label ORDER BY avg DESC'
        ).fetchall()
        conn.close()
        print(f'✅ Re-scored {len(updates)} edits with real ML model')
        print('\nNew risk distribution:')
        for r in dist:
            print(f'  {r[0]:6s}: {r[1]:3d} edits   avg score = {r[2]}')

In [ ]:
# Visualise the new score distribution
conn   = sqlite3.connect(str(DB_PATH))
scores = [r[0] for r in conn.execute('SELECT risk_score FROM predictions WHERE risk_score IS NOT NULL').fetchall()]
conn.close()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(scores, bins=40, color='#6366f1', edgecolor='white', linewidth=0.4)
ax.axvline(0.68, color='#dc2626', linestyle='--', linewidth=1.5, label='HIGH threshold (0.68)')
ax.axvline(0.30, color='#d97706', linestyle='--', linewidth=1.5, label='MEDIUM threshold (0.30)')
ax.set_xlabel('ML Risk Score (probability of damaging edit)')
ax.set_ylabel('Count')
ax.set_title('Risk Score Distribution — Real ML Scores on Dashboard Edits')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Summary

In [ ]:
print('=' * 60)
print('WikiRisk — Training Pipeline Summary')
print('=' * 60)
print(f'Data source    : Wikimedia mediawiki_history (2022–2024)')
print(f'Label          : revision_is_identity_reverted (community ground truth)')
print(f'Training rows  : {n_train:,}')
print(f'Test rows      : {n_test:,}')
print(f'Positive rate  : {100*label_stats["positive_rate"]:.1f}% (reverted edits)')
print(f'Algorithm      : SparkML LogisticRegression + TF-IDF')
print(f'AUC-ROC        : {metrics.get("auc_roc","(from manifest)")}')
print(f'F1 Score       : {metrics.get("f1","(from manifest)")}')
print(f'Precision      : {metrics.get("precision","(from manifest)")}')
print(f'Recall         : {metrics.get("recall","(from manifest)")}')
print(f'Training time  : {metrics.get("training_time_seconds",0):.0f}s')
print(f'Model path     : {model_path}')
print(f'MLflow run     : {RUN_ID}')
print('=' * 60)
print(f'Idempotent    : re-running skips all expensive steps ✅')
print(f'All artifacts  : stored in project dir (safe to zip & share) ✅')
print(f'Dashboard edits re-scored with real ML model ✅')
print(f'Streaming processor will auto-load the model on next start ✅')

if spark is not None:
    spark.stop()
    print('Spark stopped ✅')
else:
    print('Spark was not needed this run (loaded from cache) ✅')